In [1]:
# ============================================================
# ZOMBIE EXPIRED EVENT AUDIT
#
# Goal:
# 1) Check event_type của các event xảy ra SAU ngày post hết hạn
# 2) Check contact zombie:
#    - item nào hết hạn nhưng vẫn có contact
#    - seller nào có nhiều zombie posts
#    - seller đó có tổng bao nhiêu posts
#
# Default zombie logic:
#   event.date > dim_listing.expected_expired_date
#
# Nếu muốn tính luôn đúng ngày expected_expired_date là zombie:
#   COUNT_EXPIRED_DATE_AS_ZOMBIE = True
#
# Output CSV nằm trong:
#   /kaggle/working/zombie_expired_event_audit/
# ============================================================

import os
import gc
import sys
import shutil
import subprocess
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

# ============================================================
# 0. CONFIG
# ============================================================

BASE_PATH_DIM = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-1"
BASE_PATH_EVENTS = "/kaggle/input/datasets/kimkhanhnguyen214/datathon2026-2"

DIM_LISTING_PATH = f"{BASE_PATH_DIM}/dim_listing"
EVENTS_PATH = f"{BASE_PATH_EVENTS}/fact_user_events"

OUTPUT_DIR = "/kaggle/working/zombie_expired_event_audit"
ZOMBIE_PART_DIR = os.path.join(OUTPUT_DIR, "zombie_event_parts")

os.makedirs(OUTPUT_DIR, exist_ok=True)

FORCE_REBUILD = True

BATCH_SIZE_EVENTS = 300_000
BATCH_SIZE_DIM = 1_000_000

COUNT_EXPIRED_DATE_AS_ZOMBIE = False

CONTACT_EVENT_TYPES = {
    "view_phone",
    "contact_chat",
    "contact_zalo",
    "contact_sms",
    "other_interaction",
}

SAVE_ONLY_CONTACT_ZOMBIE_ROWS = False

DUCKDB_MEMORY_LIMIT = "3GB"


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)

# ============================================================
# 1. HELPERS
# ============================================================

def display_top(df, n=20, title=None):
    if title:
        print("\n" + "=" * 120)
        print(title)
        print("=" * 120)

    if df is None or len(df) == 0:
        print("Không có dữ liệu.")
        return

    try:
        display(df.head(n))
    except Exception:
        print(df.head(n))


def save_csv(df, filename):
    path = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {path} | shape={df.shape}")
    return path


def bool_to_int_series(s):
    if s is None:
        return pd.Series(dtype="int8")

    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False).astype("int8")

    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce").fillna(0).astype("int8")

    return (
        s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes", "y", "t"])
        .astype("int8")
    )


def clean_id_series(s, unknown="UNKNOWN"):
    return s.where(s.notna(), unknown).astype(str).str.strip().replace("", unknown)


def normalize_date_series(s):
    return pd.to_datetime(s, errors="coerce").dt.normalize()


def duckdb_copy_to_csv(con, query, out_path):
    out_path = out_path.replace("\\", "/")
    query = query.strip().rstrip(";")
    con.execute(f"""
        COPY (
            {query}
        )
        TO '{out_path}'
        (HEADER, DELIMITER ',')
    """)
    print("[SAVE]", out_path)
    return out_path


# ============================================================
# 2. CLEAN CACHE
# ============================================================

dim_cache_path = os.path.join(OUTPUT_DIR, "dim_item_seller_expiry_map.parquet")

if FORCE_REBUILD:
    if os.path.exists(dim_cache_path):
        os.remove(dim_cache_path)

    if os.path.exists(ZOMBIE_PART_DIR):
        shutil.rmtree(ZOMBIE_PART_DIR)

os.makedirs(ZOMBIE_PART_DIR, exist_ok=True)

# ============================================================
# 3. READ dim_listing LIGHT MAP
# ============================================================

if os.path.exists(dim_cache_path):
    print("[SKIP] dim cache exists.")
    dim_map = pd.read_parquet(dim_cache_path)

else:
    print("[BUILD] reading dim_listing light map...")

    dim_ds = ds.dataset(DIM_LISTING_PATH, format="parquet")
    available_cols = set(dim_ds.schema.names)

    wanted_cols = [
        "item_id",
        "seller_id",
        "category",
        "seller_type",
        "ad_type",
        "ad_status",
        "city_name",
        "district_name",
        "posted_date",
        "expected_expired_date",
    ]

    cols = [c for c in wanted_cols if c in available_cols]

    if "item_id" not in cols:
        raise ValueError("dim_listing thiếu item_id.")
    if "seller_id" not in cols:
        raise ValueError("dim_listing thiếu seller_id.")
    if "expected_expired_date" not in cols:
        raise ValueError("dim_listing thiếu expected_expired_date.")

    parts = []

    for i, batch in enumerate(
        dim_ds.to_batches(columns=cols, batch_size=BATCH_SIZE_DIM),
        start=1
    ):
        df = batch.to_pandas()

        for c in wanted_cols:
            if c not in df.columns:
                df[c] = np.nan

        df = df[df["item_id"].notna()].copy()

        df["item_id"] = clean_id_series(df["item_id"])
        df["seller_id"] = clean_id_series(df["seller_id"], "UNKNOWN_SELLER")

        df["posted_date"] = normalize_date_series(df["posted_date"])
        df["expected_expired_date"] = normalize_date_series(df["expected_expired_date"])

        df["event_expired_ref_date"] = df["expected_expired_date"]

        df["seller_type"] = df["seller_type"].astype(str).str.strip().str.lower()
        df["ad_type"] = df["ad_type"].astype(str).str.strip().str.lower()
        df["ad_status"] = df["ad_status"].astype(str).str.strip().str.lower()

        parts.append(df[wanted_cols + ["event_expired_ref_date"]])

        if i % 10 == 0:
            print("dim batches:", i)

        del df
        gc.collect()

    dim_map = (
        pd.concat(parts, ignore_index=True)
        .drop_duplicates("item_id")
        .reset_index(drop=True)
    )

    dim_map.to_parquet(dim_cache_path, index=False)

    del parts
    gc.collect()

print("dim_map shape:", dim_map.shape)
display_top(dim_map, 5, "dim_map sample")

dim_dq = pd.DataFrame([{
    "dim_items": dim_map["item_id"].nunique(),
    "dim_sellers": dim_map["seller_id"].nunique(),
    "items_missing_expected_expired_date": int(dim_map["expected_expired_date"].isna().sum()),
    "items_with_expected_expired_date": int(dim_map["expected_expired_date"].notna().sum()),
}])

display_top(dim_dq, title="DIM DQ")
save_csv(dim_dq, "00_dim_dq.csv")

# Seller total posts from dim_listing
seller_total_posts = (
    dim_map.groupby("seller_id", as_index=False)
    .agg(
        seller_total_posts=("item_id", "nunique"),
        seller_type_sample=("seller_type", lambda x: x.dropna().mode().iloc[0] if len(x.dropna().mode()) else "unknown"),
    )
)

seller_total_posts_path = os.path.join(OUTPUT_DIR, "seller_total_posts.parquet")
seller_total_posts.to_parquet(seller_total_posts_path, index=False)

display_top(
    seller_total_posts.sort_values("seller_total_posts", ascending=False),
    20,
    "Top sellers by total posts"
)

# Only items with valid expiry date can be used for zombie check
item_lookup = dim_map[
    dim_map["event_expired_ref_date"].notna()
][
    [
        "item_id",
        "seller_id",
        "category",
        "seller_type",
        "ad_type",
        "ad_status",
        "city_name",
        "district_name",
        "posted_date",
        "expected_expired_date",
        "event_expired_ref_date",
    ]
].copy()

valid_item_set = set(item_lookup["item_id"].unique())

print("valid items for zombie event scan:", len(valid_item_set))

# ============================================================
# 4. SCAN fact_user_events MEMORY-SAFE
# ============================================================

zombie_parts = []
dq = {
    "total_event_rows": 0,
    "missing_item_id_rows": 0,
    "missing_date_rows": 0,
    "rows_after_item_filter": 0,
    "zombie_rows_all_events": 0,
    "zombie_rows_contact_like": 0,
    "partial_files": 0,
}

if FORCE_REBUILD or len(os.listdir(ZOMBIE_PART_DIR)) == 0:
    print("[BUILD] scanning fact_user_events for expired-date zombie events...")

    event_ds = ds.dataset(EVENTS_PATH, format="parquet")
    available_cols = set(event_ds.schema.names)

    wanted_cols = [
        "user_id",
        "session_id",
        "event_id",
        "item_id",
        "event_type",
        "date",
        "event_ts",
        "is_contact",
        "surface",
        "device",
        "position",
    ]

    cols = [c for c in wanted_cols if c in available_cols]

    if "item_id" not in cols:
        raise ValueError("fact_user_events thiếu item_id.")
    if "event_type" not in cols:
        raise ValueError("fact_user_events thiếu event_type.")
    if "date" not in cols and "event_ts" not in cols:
        raise ValueError("fact_user_events thiếu cả date và event_ts.")

    part_id = 0

    for i, batch in enumerate(
        event_ds.to_batches(columns=cols, batch_size=BATCH_SIZE_EVENTS),
        start=1
    ):
        df = batch.to_pandas()

        for c in wanted_cols:
            if c not in df.columns:
                df[c] = np.nan

        dq["total_event_rows"] += len(df)
        dq["missing_item_id_rows"] += int(df["item_id"].isna().sum())

        df = df[df["item_id"].notna()].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["item_id"] = clean_id_series(df["item_id"])

        # Critical reduction before merge
        df = df[df["item_id"].isin(valid_item_set)].copy()
        dq["rows_after_item_filter"] += len(df)

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # Date
        if "date" in df.columns:
            df["date"] = normalize_date_series(df["date"])
        else:
            df["date"] = pd.NaT

        # Fallback from event_ts
        missing_date_mask = df["date"].isna()
        if missing_date_mask.any() and "event_ts" in df.columns:
            df.loc[missing_date_mask, "date"] = normalize_date_series(
                df.loc[missing_date_mask, "event_ts"]
            )

        dq["missing_date_rows"] += int(df["date"].isna().sum())

        df = df[df["date"].notna()].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        df["user_id"] = clean_id_series(df["user_id"], "UNKNOWN_USER")
        df["session_id"] = clean_id_series(df["session_id"], "UNKNOWN_SESSION")
        df["event_id"] = clean_id_series(df["event_id"], "UNKNOWN_EVENT")

        df["event_type"] = (
            df["event_type"]
            .astype(str)
            .str.strip()
            .str.lower()
            .replace("", "unknown")
        )

        # Join with expiry info
        df = df.merge(
            item_lookup,
            on="item_id",
            how="inner"
        )

        df["days_after_expired"] = (
            df["date"] - df["event_expired_ref_date"]
        ).dt.days

        if COUNT_EXPIRED_DATE_AS_ZOMBIE:
            zombie_mask = df["days_after_expired"] >= 0
        else:
            zombie_mask = df["days_after_expired"] > 0

        df = df[zombie_mask].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        # Contact flags
        if "is_contact" in df.columns:
            df["is_contact_raw"] = bool_to_int_series(df["is_contact"])
        else:
            df["is_contact_raw"] = 0

        df["is_contact_event_type"] = df["event_type"].isin(CONTACT_EVENT_TYPES).astype("int8")

        df["is_contact_like"] = (
            (df["is_contact_raw"] == 1)
            | (df["is_contact_event_type"] == 1)
        ).astype("int8")

        dq["zombie_rows_all_events"] += len(df)
        dq["zombie_rows_contact_like"] += int(df["is_contact_like"].sum())

        if SAVE_ONLY_CONTACT_ZOMBIE_ROWS:
            df = df[df["is_contact_like"] == 1].copy()

        if len(df) == 0:
            del df
            gc.collect()
            continue

        keep_cols = [
            "item_id",
            "seller_id",
            "user_id",
            "session_id",
            "event_id",
            "date",
            "event_type",
            "is_contact_raw",
            "is_contact_event_type",
            "is_contact_like",
            "days_after_expired",
            "posted_date",
            "expected_expired_date",
            "event_expired_ref_date",
            "category",
            "seller_type",
            "ad_type",
            "ad_status",
            "city_name",
            "district_name",
            "surface",
            "device",
        ]

        for c in keep_cols:
            if c not in df.columns:
                df[c] = np.nan

        part_id += 1
        part_path = os.path.join(ZOMBIE_PART_DIR, f"zombie_events_part_{part_id:05d}.parquet")
        df[keep_cols].to_parquet(part_path, index=False)

        dq["partial_files"] = part_id

        if i % 10 == 0:
            print(
                "event batches:", i,
                "| raw rows:", dq["total_event_rows"],
                "| after item filter:", dq["rows_after_item_filter"],
                "| zombie rows:", dq["zombie_rows_all_events"],
                "| zombie contact-like:", dq["zombie_rows_contact_like"],
                "| parts:", part_id
            )

        del df
        gc.collect()

else:
    print("[SKIP] zombie event parts already exist.")

dq_df = pd.DataFrame([dq])
display_top(dq_df, title="Zombie scan DQ")
save_csv(dq_df, "01_zombie_scan_dq.csv")

# ============================================================
# 5. AGGREGATE WITH DUCKDB
# ============================================================

try:
    import duckdb
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

zombie_files = [
    os.path.join(ZOMBIE_PART_DIR, f)
    for f in os.listdir(ZOMBIE_PART_DIR)
    if f.endswith(".parquet")
]

if len(zombie_files) == 0:
    print("\nKhông tìm thấy zombie event nào theo logic hiện tại.")
    print("Bạn có thể thử set COUNT_EXPIRED_DATE_AS_ZOMBIE = True rồi chạy lại.")
else:
    print("Zombie partial parquet files:", len(zombie_files))

    duckdb_path = os.path.join(OUTPUT_DIR, "zombie_audit.duckdb")
    duckdb_temp_dir = os.path.join(OUTPUT_DIR, "duckdb_temp")
    os.makedirs(duckdb_temp_dir, exist_ok=True)

    if FORCE_REBUILD and os.path.exists(duckdb_path):
        os.remove(duckdb_path)

    con = duckdb.connect(database=duckdb_path)
    con.execute("PRAGMA threads=4")
    con.execute(f"PRAGMA memory_limit='{DUCKDB_MEMORY_LIMIT}'")

    try:
        con.execute(f"SET temp_directory='{duckdb_temp_dir}'")
    except Exception:
        pass

    zombie_glob = os.path.join(ZOMBIE_PART_DIR, "*.parquet").replace("\\", "/")
    seller_total_posts_path_sql = seller_total_posts_path.replace("\\", "/")
    dim_cache_path_sql = dim_cache_path.replace("\\", "/")

    # ------------------------------------------------------------
    # 5.1 Overall summary
    # ------------------------------------------------------------

    overall_query = f"""
        SELECT
            COUNT(*) AS zombie_events_all,
            COUNT(DISTINCT item_id) AS zombie_items_all,
            COUNT(DISTINCT seller_id) AS zombie_sellers_all,
            COUNT(DISTINCT user_id) AS zombie_users_all,

            SUM(is_contact_like) AS zombie_contact_like_events,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN item_id END) AS zombie_contact_like_items,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN seller_id END) AS zombie_contact_like_sellers,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN user_id END) AS zombie_contact_like_users,

            MIN(date) AS first_zombie_event_date,
            MAX(date) AS last_zombie_event_date,

            MIN(days_after_expired) AS min_days_after_expired,
            AVG(days_after_expired) AS avg_days_after_expired,
            MAX(days_after_expired) AS max_days_after_expired
        FROM read_parquet('{zombie_glob}')
    """

    overall_csv = os.path.join(OUTPUT_DIR, "02_overall_zombie_summary.csv")
    duckdb_copy_to_csv(con, overall_query, overall_csv)

    overall_df = pd.read_csv(overall_csv)
    display_top(overall_df, title="Overall zombie summary")

    # ------------------------------------------------------------
    # 5.2 Event type summary after expired
    # ------------------------------------------------------------

    event_type_query = f"""
        SELECT
            event_type,

            COUNT(*) AS zombie_events,
            COUNT(DISTINCT item_id) AS zombie_items,
            COUNT(DISTINCT seller_id) AS zombie_sellers,
            COUNT(DISTINCT user_id) AS zombie_users,

            SUM(is_contact_raw) AS raw_is_contact_events,
            SUM(is_contact_event_type) AS contact_event_type_events,
            SUM(is_contact_like) AS contact_like_events,

            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_of_all_zombie_events,

            MIN(days_after_expired) AS min_days_after_expired,
            AVG(days_after_expired) AS avg_days_after_expired,
            MAX(days_after_expired) AS max_days_after_expired

        FROM read_parquet('{zombie_glob}')
        GROUP BY event_type
        ORDER BY zombie_events DESC
    """

    event_type_csv = os.path.join(OUTPUT_DIR, "03_zombie_event_type_summary_all_events.csv")
    duckdb_copy_to_csv(con, event_type_query, event_type_csv)

    event_type_df = pd.read_csv(event_type_csv)
    display_top(event_type_df, 30, "Event type summary AFTER expired date")

    # ------------------------------------------------------------
    # 5.3 Contact-like event type summary only
    # ------------------------------------------------------------

    contact_event_type_query = f"""
        SELECT
            event_type,

            COUNT(*) AS zombie_contact_events,
            COUNT(DISTINCT item_id) AS zombie_contact_items,
            COUNT(DISTINCT seller_id) AS zombie_contact_sellers,
            COUNT(DISTINCT user_id) AS zombie_contact_users,

            ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS pct_of_zombie_contact_events,

            MIN(days_after_expired) AS min_days_after_expired,
            AVG(days_after_expired) AS avg_days_after_expired,
            MAX(days_after_expired) AS max_days_after_expired

        FROM read_parquet('{zombie_glob}')
        WHERE is_contact_like = 1
        GROUP BY event_type
        ORDER BY zombie_contact_events DESC
    """

    contact_event_type_csv = os.path.join(OUTPUT_DIR, "04_zombie_contact_event_type_summary.csv")
    duckdb_copy_to_csv(con, contact_event_type_query, contact_event_type_csv)

    contact_event_type_df = pd.read_csv(contact_event_type_csv)
    display_top(contact_event_type_df, 30, "Contact-like event type summary AFTER expired date")

    # ------------------------------------------------------------
    # 5.4 Days-after-expired bucket x event_type
    # ------------------------------------------------------------

    bucket_event_type_query = f"""
        WITH z AS (
            SELECT
                *,
                CASE
                    WHEN days_after_expired = 0 THEN 'D0_same_expired_date'
                    WHEN days_after_expired = 1 THEN 'D1'
                    WHEN days_after_expired BETWEEN 2 AND 3 THEN 'D2_D3'
                    WHEN days_after_expired BETWEEN 4 AND 7 THEN 'D4_D7'
                    WHEN days_after_expired BETWEEN 8 AND 14 THEN 'D8_D14'
                    WHEN days_after_expired BETWEEN 15 AND 30 THEN 'D15_D30'
                    WHEN days_after_expired BETWEEN 31 AND 60 THEN 'D31_D60'
                    ELSE 'D61_plus'
                END AS days_after_expired_bucket
            FROM read_parquet('{zombie_glob}')
        )

        SELECT
            days_after_expired_bucket,
            event_type,

            COUNT(*) AS zombie_events,
            COUNT(DISTINCT item_id) AS zombie_items,
            COUNT(DISTINCT seller_id) AS zombie_sellers,
            COUNT(DISTINCT user_id) AS zombie_users,

            SUM(is_contact_like) AS contact_like_events

        FROM z
        GROUP BY days_after_expired_bucket, event_type
        ORDER BY
            CASE days_after_expired_bucket
                WHEN 'D0_same_expired_date' THEN 0
                WHEN 'D1' THEN 1
                WHEN 'D2_D3' THEN 2
                WHEN 'D4_D7' THEN 3
                WHEN 'D8_D14' THEN 4
                WHEN 'D15_D30' THEN 5
                WHEN 'D31_D60' THEN 6
                ELSE 7
            END,
            zombie_events DESC
    """

    bucket_event_type_csv = os.path.join(OUTPUT_DIR, "05_zombie_event_type_by_days_after_expired_bucket.csv")
    duckdb_copy_to_csv(con, bucket_event_type_query, bucket_event_type_csv)

    bucket_event_type_df = pd.read_csv(bucket_event_type_csv)
    display_top(bucket_event_type_df, 50, "Zombie event_type by days-after-expired bucket")

    # ------------------------------------------------------------
    # 5.5 Daily trend
    # ------------------------------------------------------------

    daily_query = f"""
        SELECT
            date,
            event_type,

            COUNT(*) AS zombie_events,
            COUNT(DISTINCT item_id) AS zombie_items,
            COUNT(DISTINCT seller_id) AS zombie_sellers,
            COUNT(DISTINCT user_id) AS zombie_users,

            SUM(is_contact_like) AS contact_like_events

        FROM read_parquet('{zombie_glob}')
        GROUP BY date, event_type
        ORDER BY date, zombie_events DESC
    """

    daily_csv = os.path.join(OUTPUT_DIR, "06_zombie_daily_event_type_trend.csv")
    duckdb_copy_to_csv(con, daily_query, daily_csv)

    daily_df = pd.read_csv(daily_csv)
    display_top(daily_df, 50, "Daily zombie event_type trend")

    # ------------------------------------------------------------
    # 5.6 Item summary: item nào hết hạn nhưng vẫn có event/contact
    # ------------------------------------------------------------

    item_summary_query = f"""
        SELECT
            item_id,
            seller_id,

            MIN(expected_expired_date) AS expected_expired_date,
            MIN(posted_date) AS posted_date,

            ANY_VALUE(category) AS category,
            ANY_VALUE(seller_type) AS seller_type,
            ANY_VALUE(ad_type) AS ad_type,
            ANY_VALUE(ad_status) AS ad_status,
            ANY_VALUE(city_name) AS city_name,
            ANY_VALUE(district_name) AS district_name,

            COUNT(*) AS zombie_events,
            SUM(is_contact_like) AS zombie_contact_events,

            COUNT(DISTINCT user_id) AS zombie_users,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN user_id END) AS zombie_contact_users,

            COUNT(DISTINCT date) AS zombie_event_days,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN date END) AS zombie_contact_days,

            MIN(date) AS first_zombie_event_date,
            MAX(date) AS last_zombie_event_date,

            MIN(CASE WHEN is_contact_like = 1 THEN date END) AS first_zombie_contact_date,
            MAX(CASE WHEN is_contact_like = 1 THEN date END) AS last_zombie_contact_date,

            MIN(days_after_expired) AS min_days_after_expired,
            AVG(days_after_expired) AS avg_days_after_expired,
            MAX(days_after_expired) AS max_days_after_expired

        FROM read_parquet('{zombie_glob}')
        GROUP BY item_id, seller_id
        ORDER BY zombie_contact_events DESC, zombie_events DESC
    """

    item_summary_csv = os.path.join(OUTPUT_DIR, "07_zombie_item_summary.csv")
    duckdb_copy_to_csv(con, item_summary_query, item_summary_csv)

    item_summary_df = pd.read_csv(item_summary_csv)
    display_top(item_summary_df, 30, "Top zombie items")

    # ------------------------------------------------------------
    # 5.7 Seller summary: seller nào có post zombie-contact
    # ------------------------------------------------------------

    seller_contact_summary_query = f"""
        WITH z_contact_item AS (
            SELECT
                seller_id,
                item_id,

                COUNT(*) AS zombie_contact_events,
                COUNT(DISTINCT user_id) AS zombie_contact_users,
                COUNT(DISTINCT date) AS zombie_contact_days,

                MIN(date) AS first_zombie_contact_date,
                MAX(date) AS last_zombie_contact_date,

                MIN(days_after_expired) AS min_days_after_expired,
                AVG(days_after_expired) AS avg_days_after_expired,
                MAX(days_after_expired) AS max_days_after_expired

            FROM read_parquet('{zombie_glob}')
            WHERE is_contact_like = 1
            GROUP BY seller_id, item_id
        ),

        seller_zombie AS (
            SELECT
                seller_id,

                COUNT(DISTINCT item_id) AS zombie_contact_posts,
                SUM(zombie_contact_events) AS zombie_contact_events,
                SUM(zombie_contact_users) AS zombie_contact_users_sum_by_item,
                SUM(zombie_contact_days) AS zombie_contact_days_sum_by_item,

                MIN(first_zombie_contact_date) AS first_zombie_contact_date,
                MAX(last_zombie_contact_date) AS last_zombie_contact_date,

                MIN(min_days_after_expired) AS min_days_after_expired,
                AVG(avg_days_after_expired) AS avg_days_after_expired,
                MAX(max_days_after_expired) AS max_days_after_expired

            FROM z_contact_item
            GROUP BY seller_id
        ),

        seller_totals AS (
            SELECT
                seller_id,
                seller_total_posts,
                seller_type_sample
            FROM read_parquet('{seller_total_posts_path_sql}')
        )

        SELECT
            sz.seller_id,

            sz.zombie_contact_posts,
            st.seller_total_posts,

            sz.zombie_contact_posts * 1.0 / NULLIF(st.seller_total_posts, 0) AS zombie_contact_post_rate,

            sz.zombie_contact_events,
            sz.zombie_contact_users_sum_by_item,
            sz.zombie_contact_days_sum_by_item,

            sz.first_zombie_contact_date,
            sz.last_zombie_contact_date,

            sz.min_days_after_expired,
            sz.avg_days_after_expired,
            sz.max_days_after_expired,

            st.seller_type_sample

        FROM seller_zombie sz
        LEFT JOIN seller_totals st
            ON sz.seller_id = st.seller_id

        ORDER BY
            sz.zombie_contact_posts DESC,
            zombie_contact_post_rate DESC,
            sz.zombie_contact_events DESC
    """

    seller_contact_summary_csv = os.path.join(OUTPUT_DIR, "08_zombie_seller_contact_summary.csv")
    duckdb_copy_to_csv(con, seller_contact_summary_query, seller_contact_summary_csv)

    seller_contact_summary_df = pd.read_csv(seller_contact_summary_csv)
    display_top(seller_contact_summary_df, 50, "Top sellers by zombie-contact posts")

    # ------------------------------------------------------------
    # 5.8 Seller summary all zombie events, not only contact
    # ------------------------------------------------------------

    seller_all_summary_query = f"""
        WITH z_item AS (
            SELECT
                seller_id,
                item_id,

                COUNT(*) AS zombie_events,
                SUM(is_contact_like) AS zombie_contact_events,

                COUNT(DISTINCT user_id) AS zombie_users,
                COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN user_id END) AS zombie_contact_users,

                COUNT(DISTINCT date) AS zombie_event_days,
                COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN date END) AS zombie_contact_days,

                MIN(date) AS first_zombie_event_date,
                MAX(date) AS last_zombie_event_date,

                MIN(days_after_expired) AS min_days_after_expired,
                AVG(days_after_expired) AS avg_days_after_expired,
                MAX(days_after_expired) AS max_days_after_expired

            FROM read_parquet('{zombie_glob}')
            GROUP BY seller_id, item_id
        ),

        seller_zombie AS (
            SELECT
                seller_id,

                COUNT(DISTINCT item_id) AS zombie_any_event_posts,
                SUM(CASE WHEN zombie_contact_events > 0 THEN 1 ELSE 0 END) AS zombie_contact_posts,

                SUM(zombie_events) AS zombie_events,
                SUM(zombie_contact_events) AS zombie_contact_events,

                SUM(zombie_users) AS zombie_users_sum_by_item,
                SUM(zombie_contact_users) AS zombie_contact_users_sum_by_item,

                SUM(zombie_event_days) AS zombie_event_days_sum_by_item,
                SUM(zombie_contact_days) AS zombie_contact_days_sum_by_item,

                MIN(first_zombie_event_date) AS first_zombie_event_date,
                MAX(last_zombie_event_date) AS last_zombie_event_date,

                MIN(min_days_after_expired) AS min_days_after_expired,
                AVG(avg_days_after_expired) AS avg_days_after_expired,
                MAX(max_days_after_expired) AS max_days_after_expired

            FROM z_item
            GROUP BY seller_id
        ),

        seller_totals AS (
            SELECT
                seller_id,
                seller_total_posts,
                seller_type_sample
            FROM read_parquet('{seller_total_posts_path_sql}')
        )

        SELECT
            sz.seller_id,

            sz.zombie_any_event_posts,
            sz.zombie_contact_posts,
            st.seller_total_posts,

            sz.zombie_any_event_posts * 1.0 / NULLIF(st.seller_total_posts, 0) AS zombie_any_event_post_rate,
            sz.zombie_contact_posts * 1.0 / NULLIF(st.seller_total_posts, 0) AS zombie_contact_post_rate,

            sz.zombie_events,
            sz.zombie_contact_events,

            sz.zombie_users_sum_by_item,
            sz.zombie_contact_users_sum_by_item,

            sz.zombie_event_days_sum_by_item,
            sz.zombie_contact_days_sum_by_item,

            sz.first_zombie_event_date,
            sz.last_zombie_event_date,

            sz.min_days_after_expired,
            sz.avg_days_after_expired,
            sz.max_days_after_expired,

            st.seller_type_sample

        FROM seller_zombie sz
        LEFT JOIN seller_totals st
            ON sz.seller_id = st.seller_id

        ORDER BY
            sz.zombie_contact_posts DESC,
            sz.zombie_any_event_posts DESC,
            zombie_contact_post_rate DESC,
            sz.zombie_contact_events DESC
    """

    seller_all_summary_csv = os.path.join(OUTPUT_DIR, "09_zombie_seller_all_event_summary.csv")
    duckdb_copy_to_csv(con, seller_all_summary_query, seller_all_summary_csv)

    seller_all_summary_df = pd.read_csv(seller_all_summary_csv)
    display_top(seller_all_summary_df, 50, "Top sellers by all zombie events")

    # ------------------------------------------------------------
    # 5.9 Seller x item contact detail
    # ------------------------------------------------------------

    seller_item_contact_query = f"""
        WITH z_contact_item AS (
            SELECT
                seller_id,
                item_id,

                COUNT(*) AS zombie_contact_events,
                COUNT(DISTINCT user_id) AS zombie_contact_users,
                COUNT(DISTINCT date) AS zombie_contact_days,

                STRING_AGG(DISTINCT event_type, ', ' ORDER BY event_type) AS zombie_contact_event_types,

                MIN(date) AS first_zombie_contact_date,
                MAX(date) AS last_zombie_contact_date,

                MIN(days_after_expired) AS min_days_after_expired,
                AVG(days_after_expired) AS avg_days_after_expired,
                MAX(days_after_expired) AS max_days_after_expired

            FROM read_parquet('{zombie_glob}')
            WHERE is_contact_like = 1
            GROUP BY seller_id, item_id
        ),

        seller_totals AS (
            SELECT
                seller_id,
                seller_total_posts
            FROM read_parquet('{seller_total_posts_path_sql}')
        ),

        item_meta AS (
            SELECT
                item_id,
                seller_id,
                category,
                seller_type,
                ad_type,
                ad_status,
                city_name,
                district_name,
                posted_date,
                expected_expired_date
            FROM read_parquet('{dim_cache_path_sql}')
        )

        SELECT
            zci.seller_id,
            st.seller_total_posts,

            zci.item_id,
            im.category,
            im.seller_type,
            im.ad_type,
            im.ad_status,
            im.city_name,
            im.district_name,
            im.posted_date,
            im.expected_expired_date,

            zci.zombie_contact_events,
            zci.zombie_contact_users,
            zci.zombie_contact_days,
            zci.zombie_contact_event_types,

            zci.first_zombie_contact_date,
            zci.last_zombie_contact_date,

            zci.min_days_after_expired,
            zci.avg_days_after_expired,
            zci.max_days_after_expired

        FROM z_contact_item zci
        LEFT JOIN seller_totals st
            ON zci.seller_id = st.seller_id
        LEFT JOIN item_meta im
            ON zci.item_id = im.item_id

        ORDER BY
            zci.zombie_contact_events DESC,
            zci.zombie_contact_users DESC,
            zci.max_days_after_expired DESC
    """

    seller_item_contact_csv = os.path.join(OUTPUT_DIR, "10_zombie_seller_item_contact_detail.csv")
    duckdb_copy_to_csv(con, seller_item_contact_query, seller_item_contact_csv)

    seller_item_contact_df = pd.read_csv(seller_item_contact_csv)
    display_top(seller_item_contact_df, 50, "Zombie seller-item contact detail")

    # ------------------------------------------------------------
    # 5.10 Category / seller_type summary
    # ------------------------------------------------------------

    category_summary_query = f"""
        SELECT
            category,
            seller_type,
            ad_type,

            COUNT(*) AS zombie_events,
            SUM(is_contact_like) AS zombie_contact_events,

            COUNT(DISTINCT item_id) AS zombie_items,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN item_id END) AS zombie_contact_items,

            COUNT(DISTINCT seller_id) AS zombie_sellers,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN seller_id END) AS zombie_contact_sellers,

            COUNT(DISTINCT user_id) AS zombie_users,
            COUNT(DISTINCT CASE WHEN is_contact_like = 1 THEN user_id END) AS zombie_contact_users,

            MIN(days_after_expired) AS min_days_after_expired,
            AVG(days_after_expired) AS avg_days_after_expired,
            MAX(days_after_expired) AS max_days_after_expired

        FROM read_parquet('{zombie_glob}')
        GROUP BY category, seller_type, ad_type
        ORDER BY zombie_contact_events DESC, zombie_events DESC
    """

    category_summary_csv = os.path.join(OUTPUT_DIR, "11_zombie_category_seller_type_summary.csv")
    duckdb_copy_to_csv(con, category_summary_query, category_summary_csv)

    category_summary_df = pd.read_csv(category_summary_csv)
    display_top(category_summary_df, 50, "Zombie by category / seller_type / ad_type")

    con.close()

    print("\nDONE.")
    print("Output folder:", OUTPUT_DIR)
    print("\nFile quan trọng nhất nên xem:")
    print("03_zombie_event_type_summary_all_events.csv")
    print("04_zombie_contact_event_type_summary.csv")
    print("08_zombie_seller_contact_summary.csv")
    print("10_zombie_seller_item_contact_detail.csv")

[BUILD] reading dim_listing light map...
dim batches: 10
dim batches: 20
dim batches: 30
dim batches: 40
dim_map shape: (3107114, 11)

dim_map sample


,item_id,seller_id,category,seller_type,ad_type,ad_status,city_name,district_name,posted_date,expected_expired_date,event_expired_ref_date
0,bc8a95ba9c8c41b86fc446e58af07aacf1e98cb0a8f63253489f4d150d0b79d5,07d8a5ec937d46f94da7d680aa3f8304bfc84b7c3749357b2bebacffda8cfbd9,1010,agent,let,deleted,Tp Hồ Chí Minh,Quận 3,2025-03-04,2025-03-19,2025-03-19
1,2e58d907f4986a98aa31700ae56883bbc85748932d58afbc4b634a00721091d5,baf1308905d9874c49d73dbccce137fa6715dbd515d3ec92ef86e52c0f2fb19c,1010,agent,let,deleted,Tp Hồ Chí Minh,Quận 8,2025-05-04,2025-05-19,2025-05-19
2,3d186c9e829ba17729a239816bcfe24c9e23bdd119f62c3b5b89402e759de792,b7b7612ad20cbbea6cfb76c41f281904fdeb95ccfc8cdbe7d2eacb26123ac521,1010,agent,let,deleted,Tp Hồ Chí Minh,Quận 8,2025-04-17,2025-05-02,2025-05-02
3,e9f549a8c6b0ed5c823df7de5c0344d7b0b1777a65af61ef0ff7553f973f84df,34aaafc98adbfd5df1e13c9e8e098097822b1ea9895b0cdb10807d52e307dfea,1010,private,let,deleted,Tp Hồ Chí Minh,Quận 3,2025-07-03,2025-07-13,2025-07-13
4,6c30deece657bdce3cd0ab779e3dc0954ca775cd2753798f516f86b44ef6d23a,b99ae59154245852ea979db9fee339e41b819cc73a8b4bfb1f60299dbb97500a,1010,agent,let,deleted,Tp Hồ Chí Minh,Quận Gò Vấp,2025-11-26,2025-12-11,2025-12-11



DIM DQ


,dim_items,dim_sellers,items_missing_expected_expired_date,items_with_expected_expired_date
0,3107114,327946,0,3107114


[SAVE] /kaggle/working/zombie_expired_event_audit/00_dim_dq.csv | shape=(1, 4)

Top sellers by total posts


...

[SAVE] /kaggle/working/zombie_expired_event_audit/05_zombie_event_type_by_days_after_expired_bucket.csv

Zombie event_type by days-after-expired bucket


,days_after_expired_bucket,event_type,zombie_events,zombie_items,zombie_sellers,zombie_users,contact_like_events
0,D1,other_interaction,110029,15509,11610,17629,110029
1,D1,pageview,39402,23427,16738,28485,0
2,D1,view_phone,17774,2163,2022,2270,17774
3,D1,contact_chat,848,515,503,525,848
4,D1,contact_sms,748,245,242,258,748
5,D1,contact_zalo,458,73,72,75,458
6,D2_D3,other_interaction,193284,25624,17315,30477,193284
7,D2_D3,pageview,71693,37508,23998,48829,0
8,D2_D3,view_phone,28713,3791,3443,3904,28713
9,D2_D3,contact_chat,1401,796,769,857,1401


[SAVE] /kaggle/working/zombie_expired_event_audit/06_zombie_daily_event_type_trend.csv

Daily zombie event_type trend


,date,event_type,zombie_events,zombie_items,zombie_sellers,zombie_users,contact_like_events
0,2025-11-09 00:00:00,other_interaction,15677,3235,2753,2680,15677
1,2025-11-09 00:00:00,pageview,6589,4354,3602,3757,0
2,2025-11-09 00:00:00,view_phone,1081,262,259,226,1081
3,2025-11-09 00:00:00,contact_chat,114,74,73,72,114
4,2025-11-09 00:00:00,contact_sms,87,39,38,38,87
5,2025-11-09 00:00:00,contact_zalo,33,4,4,4,33
6,2025-11-10 00:00:00,other_interaction,20524,4131,3465,3329,20524
7,2025-11-10 00:00:00,pageview,7882,5496,4464,4437,0
8,2025-11-10 00:00:00,view_phone,1194,343,335,300,1194
9,2025-11-10 00:00:00,contact_sms,116,46,46,45,116


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))